In [3]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")

In [4]:
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
import requests

class WeatherLoader(BaseLoader):
    def __init__(self,city:str):
        self.city = city


    def load(self) -> list[Document]:
        url = (
            "https://api.open-meteo.com/v1/forecast"
            "?latitude=37.77&longitude=-122.41&current_weather=true"
        )
        response  = requests.get(url)
        if response.status_code != 200:
            raise ValueError(f"Failed to fetch weather data: {response.status_code}")

        data = response.json()
        weather = data.get("current_weather", {})
        text = self._format_weather(weather)
        return [Document(page_content=text, metadata={"source": self.city})]

    def _format_weather(self, weather: dict) -> str:
        if not weather:
            return "No weather data available."
        return (
            f"City: {self.city}\n"
            f"Temperature: {weather.get('temperature')}°C\n"
            f"Wind Speed: {weather.get('windspeed')} km/h\n"
            f"Weather Code: {weather.get('weathercode')}\n"
            f"Time: {weather.get('time')}"
        )    

In [ ]:
weather_loader  = WeatherLoader("San Franciso")
weather_docs = weather_loader.load()

In [10]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")
weather_docs
weather_retriever = FAISS.from_documents(weather_docs, embeddings).as_retriever()

In [16]:
from typing import List, TypedDict


class AgentState(TypedDict):
    messages: List
    context: str

In [23]:
from langchain_core.messages import HumanMessage


def weather_agent_node(state: AgentState):
    question = state['messages'][-1].content
    content = "\n".join([d.page_content for d in weather_retriever.invoke(question)])
    response = llm.invoke([HumanMessage(content=f"Answer weather question.\nContext: {content}\nQ: {question}")])
    return {"messages": state['messages'] + [response], "context": content}
    

In [26]:
from langgraph.graph import END, StateGraph


graph = StateGraph(AgentState)
graph.add_node("weather_agent", weather_agent_node)
graph.set_entry_point("weather_agent")
graph.add_edge("weather_agent", END)
app = graph.compile()

In [ ]:
result = app.invoke({
    "messages": [HumanMessage(content="What is the wind speed in San Francisco right now?")],
    "context": ""
})

print(result["messages"][-1].content)